
# Late Delivery Prediction — Machine Learning

## Project Overview

This project develops a **binary classification model** to predict whether an e-commerce order will arrive late using the Brazilian Olist e-commerce dataset.

The goal is to identify orders at risk of late delivery using information available **at the time of purchase**, helping businesses improve delivery reliability and customer satisfaction.

### Objectives
- Prepare and explore historical order data from PostgreSQL.
- Engineer predictive features while preventing data leakage.
- Train and compare Logistic Regression and Random Forest classifiers.
- Evaluate model performance using precision, recall, F1-score, and confusion matrices.

### Prediction Target
- **0 — Not Late:** Delivered on or before the estimated delivery date.
- **1 — Late:** Delivered after the estimated delivery date.

Orders with missing delivery outcomes will be excluded from training.


## 1. Data Extraction

Load historical order data from the PostgreSQL `retail_bi` database using Pandas and SQLAlchemy.

We'll begin with order-level information and our binary prediction target (`is_late`). Additional features will be introduced during feature engineering.

In [1]:
# Install the PostgreSQL driver into the notebook's Python environment, psycopg2 handles PostgreSQL communication
# %pip install psycopg2-binary
# pip install scikit-learn

import os
import pandas as pd
# SQLAlchemy manages database connections
from sqlalchemy import create_engine, text
from getpass import getpass
import sklearn

# Prompt for the database password without displaying it in the notebook
db_password = getpass("PostgreSQL password: ")

# Create a connection to our local PostgreSQL database
engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "host": "localhost",
        "port": 5432,
        "dbname": "retail_bi",
        "user": "postgres",
        "password": db_password
    }
)

# Execute a simple SQL query to verify the database connection
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Connection successful:", result.scalar())

Connection successful: 1


In [2]:
# SQL query to retrieve historical orders and create our prediction target
query = """
SELECT
    order_id,
    customer_id,
    order_purchase_timestamp,
    order_estimated_delivery_date,
    delivery_status,

    -- Binary classification target: 1 = Late, 0 = Not Late
    CASE
        WHEN delivery_status = 'Late' THEN 1
        ELSE 0
    END AS is_late

FROM orders

-- Exclude orders with missing delivery outcomes
WHERE delivery_status IN ('Early', 'On Time', 'Late')
"""

# Execute the SQL query and load the results into a Pandas DataFrame
df = pd.read_sql(query, engine)

# Preview the first five rows
print(df.head())

# Display the number of rows and columns in our dataset
print("Dataset shape:", df.shape)

# Count how many orders belong to each target class
print("\nTarget distribution:")
print(df["is_late"].value_counts())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_purchase_timestamp order_estimated_delivery_date delivery_status  \
0      2017-10-02 10:56:33                    2017-10-18           Early   
1      2018-07-24 20:41:37                    2018-08-13           Early   
2      2018-08-08 08:38:49                    2018-09-04           Early   
3      2017-11-18 19:28:06                    2017-12-15           Early   
4      2018-02-13 21:18:39                    2018-02-26           Early   

   is_late  
0        0  
1        0  
2        0  
3        0  
4        0  
Dataset shape: (96476, 6)


In [3]:
# Count missing values in each column
print("Missing values:")
print(df.isna().sum())

# Check whether any order_id appears more than once
print("\nDuplicate orders:", df["order_id"].duplicated().sum())

# Inspect the data types of our columns
print("\nColumn data types:")
print(df.dtypes)


Missing values:
order_id                         0
customer_id                      0
order_purchase_timestamp         0
order_estimated_delivery_date    0
delivery_status                  0
is_late                          0
dtype: int64

Duplicate orders: 0

Column data types:
order_id                                    str
customer_id                                 str
order_purchase_timestamp         datetime64[us]
order_estimated_delivery_date    datetime64[us]
delivery_status                             str
is_late                                   int64
dtype: object


## 2. Feature Engineering

Create predictive features using information available at the time of purchase.

### Purchase Timing & Delivery Estimates
- Extract purchase month, day of week, and hour.
- Calculate the estimated delivery window in days.

### Customer Location
- Retrieve customer state information from PostgreSQL.
- Merge customer data with our order-level dataset using `customer_id`.
- Check for missing values after the merge.

Customer location may help identify regional differences in delivery reliability.

In [4]:
# Extract the month when the customer placed the order (1–12)
df["purchase_month"] = df["order_purchase_timestamp"].dt.month

# Extract the day of the week (Monday = 0, Sunday = 6)
df["purchase_day_of_week"] = df["order_purchase_timestamp"].dt.dayofweek

# Extract the hour when the order was placed (0–23)
df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

# Calculate the number of days between purchase and estimated delivery
df["estimated_delivery_days"] = (
    df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400 # converts seconds into days.

# Preview our new features alongside the prediction target
df[
    [
        "purchase_month",
        "purchase_day_of_week",
        "purchase_hour",
        "estimated_delivery_days",
        "is_late"
    ]
].head()


,purchase_month,purchase_day_of_week,purchase_hour,estimated_delivery_days,is_late
0,10,0,10,15.544063,0
1,7,1,20,19.137766,0
2,8,2,8,26.639711,0
3,11,5,19,26.188819,0
4,2,1,21,12.112049,0


In [5]:
# Retrieve customer IDs and their corresponding states from PostgreSQL
customers_query = """
SELECT
    customer_id,
    customer_state
FROM customers
"""

# Load the customer data into a separate DataFrame
customers_df = pd.read_sql(customers_query, engine)

# Merge customer states into our ML dataset using customer_id
# how="left" keeps every order from our original dataset
# validate="many_to_one" checks that each customer_id matches at most one customer record
df = df.merge(
    customers_df,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Check whether any orders are missing customer state information
print("Missing customer states:", df["customer_state"].isna().sum())

# Preview our new feature
df[["customer_id", "customer_state", "is_late"]].head()


Missing customer states: 0


,customer_id,customer_state,is_late
0,9ef432eb6251297304e76186b10a928d,SP,0
1,b0830fb4747a6c6d20dea0b8c802d7ef,BA,0
2,41ce2a54c0b03bf3443c3d931a367089,GO,0
3,f88197465ea7920adcdbec7375364d82,RN,0
4,8ab97904e6daea8866dbdbc4fb7aad2c,SP,0


### Order-Level Features

Aggregate data from `fact_order_items` to create two additional predictive features:

- **`total_order_value`** — Total merchandise value of each order.
- **`total_items`** — Number of items purchased per order.

Since `fact_order_items` contains multiple rows per order, we use SQL aggregation (`GROUP BY order_id`) before merging to preserve **one row per order** in our ML dataset.

These features may help identify relationships between order size, purchase value, and delivery delays.

In [6]:
# Aggregate order items into one row per order
order_items_query = """
SELECT
    order_id,
    SUM(price) AS total_order_value,
    COUNT(*) AS total_items
FROM fact_order_items
GROUP BY order_id
"""

# Load aggregated order information from PostgreSQL
order_items_df = pd.read_sql(order_items_query, engine)

# Merge order-level features into our ML dataset
# validate="one_to_one" ensures neither DataFrame has duplicate order IDs
df = df.merge(
    order_items_df,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Check whether any orders are missing item information
print("Missing order values:", df["total_order_value"].isna().sum())
print("Missing item counts:", df["total_items"].isna().sum())

# Confirm the number of rows hasn't changed
print("Dataset shape:", df.shape)

# Preview the new features
df[["order_id", "total_order_value", "total_items", "is_late"]].head()

# Verify that every row represents a unique order
print("Duplicate orders:", df["order_id"].duplicated().sum())



Missing order values: 0
Missing item counts: 0
Dataset shape: (96476, 13)
Duplicate orders: 0


### Seller-Level Features

Investigate seller information to identify potential features related to shipping origin and delivery performance.

- Count the number of unique sellers per order using `COUNT(DISTINCT seller_id)`.
- Determine how frequently orders contain products from multiple sellers.
- Evaluate how seller location can be incorporated while preserving **one row per order**.

Seller location may help capture geographic factors affecting delivery times and delays.

In [7]:
# Count the number of unique sellers associated with each order
seller_query = """
SELECT
    order_id,
    COUNT(DISTINCT seller_id) AS total_sellers
FROM fact_order_items
GROUP BY order_id
"""

# Load seller counts from PostgreSQL
seller_counts_df = pd.read_sql(seller_query, engine)

# Count how many orders have 1, 2, 3, etc. sellers
print("Number of sellers per order:")
print(seller_counts_df["total_sellers"].value_counts().sort_index())

Number of sellers per order:
total_sellers
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64


#### Number of Sellers per Order

Approximately **98.7% of orders contain products from a single seller**, while 1.3% involve multiple sellers.

- Merge `total_sellers` into the ML dataset using `order_id`.
- Preserve **one row per order** to prevent duplicate training records.
- Validate the merge by checking for missing values and duplicate orders.

The number of sellers may help capture differences in order fulfillment complexity and delivery risk.

In [8]:
# Merge the number of unique sellers into our ML dataset
df = df.merge(
    seller_counts_df,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Verify the merge didn't introduce missing values or duplicate orders
print("Missing seller counts:", df["total_sellers"].isna().sum())
print("Duplicate orders:", df["order_id"].duplicated().sum())

# Preview our new feature
df[["order_id", "total_sellers", "is_late"]].head()

Missing seller counts: 0
Duplicate orders: 0


,order_id,total_sellers,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,1,0
1,53cdb2fc8bc7dce0b6741e2150273451,1,0
2,47770eb9100c2d0c44946d9cf07ec65d,1,0
3,949d5b44dbf5de918fe9c16f97b45f8a,1,0
4,ad21c59c0840e6cb83a9ceb5573f8159,1,0


In [9]:
# Retrieve seller location information aggregated by order
# ARRAY_AGG(DISTINCT s.seller_state) collects unique seller states into an array for each order.
seller_location_query = """
SELECT
    f.order_id,
    COUNT(DISTINCT s.seller_state) AS total_seller_states,
    ARRAY_AGG(DISTINCT s.seller_state) AS seller_states
FROM fact_order_items f
JOIN sellers s
    ON f.seller_id = s.seller_id
GROUP BY f.order_id
"""

# Execute the query and load the results into a DataFrame
seller_location_df = pd.read_sql(seller_location_query, engine)

# Merge seller locations into our order-level ML dataset
df = df.merge(
    seller_location_df,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Check for missing values and duplicate orders
print("Missing seller locations:", df["seller_states"].isna().sum())
print("Duplicate orders:", df["order_id"].duplicated().sum())

# Preview seller locations alongside customer states
print(df[["customer_state", "seller_states", "total_seller_states"]].head())

Missing seller locations: 0
Duplicate orders: 0
  customer_state seller_states  total_seller_states
0             SP          [SP]                    1
1             BA          [SP]                    1
2             GO          [SP]                    1
3             RN          [MG]                    1
4             SP          [SP]                    1


In [10]:
# Check whether every seller state matches the customer's state
# all() returns True only if every comparison is True
df["same_state_shipping"] = df.apply(
    # function lambda takes a row, returns int value of true(1) or false(0)
    lambda row: int(
        # Returns true if all the seller states matches customer state (obvs won't if theres more than 1 seller state)
        all(
            
            state == row["customer_state"] 
            # for loop to check every state in seller_state since that is an array
            for state in row["seller_states"]
        )
    ),
    axis=1 # axis=1 - apply the function to each row, which is why we can use row as a varible in our lambda function
) 

# Count orders in each shipping category
print(df["same_state_shipping"].value_counts())

# Preview our new feature
df[["customer_state", "seller_states", "same_state_shipping", "is_late"]].head()


same_state_shipping
0    61899
1    34577
Name: count, dtype: int64


,customer_state,seller_states,same_state_shipping,is_late
0,SP,[SP],1,0
1,BA,[SP],0,0
2,GO,[SP],0,0
3,RN,[MG],0,0
4,SP,[SP],1,0


In [11]:
# Compare late delivery rates for same-state vs different-state shipping
df.groupby("same_state_shipping")["is_late"].agg(["count", "sum", "mean"])

,count,sum,mean
same_state_shipping,,,
0,61899,4972,0.080324
1,34577,1563,0.045203


#### Same-State Shipping

Created `same_state_shipping` to identify whether all sellers are located in the same state as the customer.

- `1`: All sellers are in the customer's state.
- `0`: At least one seller is in a different state.

**Findings:**
- Different-state orders: 61,899 orders, with an **8.03%** late delivery rate.
- Same-state orders: 34,577 orders, with a **4.52%** late delivery rate.
- Different-state orders were approximately **1.78 times more likely** to arrive late.

This suggests that seller-customer location differences may be a useful predictor of late deliveries.

## 3. Data Preparation

Prepare the dataset for machine learning by selecting relevant features and separating the target variable.

- **Features (`X`):** Information available at purchase time that may help predict late deliveries.
- **Target (`y`):** `is_late`, where `1` indicates a late delivery and `0` indicates an early or on-time delivery.

Exclude order identifiers, actual delivery outcomes, and other columns that could cause data leakage.

In [12]:
# Select the features we'll use to predict late deliveries
feature_columns = [
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "estimated_delivery_days",
    "customer_state",
    "total_order_value",
    "total_items",
    "total_sellers",
    "total_seller_states",
    "same_state_shipping"
]

# X contains the input features
X = df[feature_columns].copy() # .copy() creates separate copies of the selected data.

# y contains the outcome we want to predict
y = df["is_late"].copy()

# Check the dimensions
print("Features:", X.shape)
print("Target:", y.shape)

print(X.head())

Features: (96476, 10)
Target: (96476,)
   purchase_month  purchase_day_of_week  purchase_hour  \
0              10                     0             10   
1               7                     1             20   
2               8                     2              8   
3              11                     5             19   
4               2                     1             21   

   estimated_delivery_days customer_state  total_order_value  total_items  \
0                15.544063             SP              29.99            1   
1                19.137766             BA             118.70            1   
2                26.639711             GO             159.90            1   
3                26.188819             RN              45.00            1   
4                12.112049             SP              19.90            1   

   total_sellers  total_seller_states  same_state_shipping  
0              1                    1                    1  
1              1           

### Train-Test Split

Split the dataset chronologically to simulate predicting future orders using historical data.

- **Training set (80%):** Used to train the machine learning model.
- **Testing set (20%):** Used to evaluate performance on unseen orders.

A chronological split helps prevent future orders from being used to predict earlier orders.

In [13]:
# Sort orders chronologically by their purchase timestamp
df_sorted = df.sort_values("order_purchase_timestamp")

# Calculate where to split the dataset (First 80% training, Last 20% testing)
split_index = int(len(df_sorted) * 0.8)

# Select the training and testing features
X_train = df_sorted[feature_columns].iloc[:split_index].copy()
X_test = df_sorted[feature_columns].iloc[split_index:].copy()

# Select the corresponding target values
y_train = df_sorted["is_late"].iloc[:split_index].copy()
y_test = df_sorted["is_late"].iloc[split_index:].copy()

# Check the sizes and late delivery rates
print("Training:", X_train.shape, y_train.shape)
print("Testing:", X_test.shape, y_test.shape)

print("Training late rate:", y_train.mean())
print("Testing late rate:", y_test.mean())

# Check the purchase date ranges in each dataset
print("Training period:")
print(df_sorted["order_purchase_timestamp"].iloc[:split_index].agg(["min", "max"]))

print("\nTesting period:")
print(df_sorted["order_purchase_timestamp"].iloc[split_index:].agg(["min", "max"]))

Training: (77180, 10) (77180,)
Testing: (19296, 10) (19296,)
Training late rate: 0.07593936252915263
Testing late rate: 0.03492951907131012
Training period:
min   2016-09-15 12:16:38
max   2018-05-26 16:54:05
Name: order_purchase_timestamp, dtype: datetime64[us]

Testing period:
min   2018-05-26 17:57:44
max   2018-08-29 15:00:37
Name: order_purchase_timestamp, dtype: datetime64[us]


In [14]:
# Convert customer states into numerical columns
X_train_encoded = pd.get_dummies( # pd.get_dummies() converts categorical values into numerical columns containing 0s and 1s.
    X_train, # The DataFrame we're converting.
    columns=["customer_state"], #Only encode this column.
    dtype=int # Use integers (0 and 1) instead of booleans (False and True).
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=["customer_state"],
    dtype=int
)

# Ensure testing has exactly the same feature columns as training, in case some state wasn't one-hot encoded
X_test_encoded = X_test_encoded.reindex( #.reindex() Rearranges, adds, or removes columns to match a specified set.
    columns=X_train_encoded.columns, # Use the training dataset's columns as the reference
    fill_value=0 #If a column is missing, create it and fill it with zeros.
)

print("Training:", X_train_encoded.shape)
print("Testing:", X_test_encoded.shape)

print(X_train_encoded.head())

Training: (77180, 36)
Testing: (19296, 36)
       purchase_month  purchase_day_of_week  purchase_hour  \
29811               9                     3             12   
90509              10                     0              9   
27599              10                     0             16   
89876              10                     0             21   
95058              10                     0             21   

       estimated_delivery_days  total_order_value  total_items  total_sellers  \
29811                18.488449             134.97            3              1   
90509                23.593866              29.90            1              1   
27599                34.293866              21.90            1              1   
89876                52.123831              21.50            1              1   
95058                56.115556              36.49            1              1   

       total_seller_states  same_state_shipping  customer_state_AC  ...  \
29811                 

### Feature Scaling — StandardScaler

Standardized numerical features using `StandardScaler` to ensure features with different numerical ranges are comparable during Logistic Regression training. Without scaling, features with larger numerical values can affect how efficiently Logistic Regression learns.

#### StandardScaler Formula

$$
z = \frac{x - \mu}{\sigma}
$$

Where:
- $z$ = Standardized value
- $x$ = Original feature value
- $\mu$ = Mean of the feature in the training dataset
- $\sigma$ = Standard deviation of the feature in the training dataset

After scaling, each numerical feature in the training dataset has approximately:
- Mean = 0
- Standard deviation = 1

**Implementation:**
- `fit_transform()` calculates the mean and standard deviation from training data and scales it.
- `transform()` applies those same training statistics to testing data.
- Using training statistics for both datasets prevents data leakage.
- Binary and one-hot encoded features remain as 0s and 1s.

In [15]:
from sklearn.preprocessing import StandardScaler

# Select numerical features that need scaling, leaving out the ones we one-hot encoded
numeric_features = [
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "estimated_delivery_days",
    "total_order_value",
    "total_items",
    "total_sellers",
    "total_seller_states"
]

# Create the scaler
scaler = StandardScaler()

# Learn the scaling parameters from training data
# fit_transform() calculates the mean and standard deviation from the training data, then scales it.
X_train_encoded[numeric_features] = scaler.fit_transform(X_train_encoded[numeric_features])

# Apply the same scaling parameters to testing data
X_test_encoded[numeric_features] = scaler.transform(
    X_test_encoded[numeric_features]
)

# Preview the standardized features
X_train_encoded[numeric_features].head()

,purchase_month,purchase_day_of_week,purchase_hour,estimated_delivery_days,total_order_value,total_items,total_sellers,total_seller_states
29811,0.901398,0.119700,-0.517978,-0.724945,-0.008169,3.403476,-0.108491,-0.068968
90509,1.183406,-1.407824,-1.078828,-0.091574,-0.518138,-0.264167,-0.108491,-0.068968
27599,1.183406,-1.407824,0.229821,1.235852,-0.556967,-0.264167,-0.108491,-0.068968
89876,1.183406,-1.407824,1.164571,3.447811,-0.558909,-0.264167,-0.108491,-0.068968
95058,1.183406,-1.407824,1.164571,3.943018,-0.486153,-0.264167,-0.108491,-0.068968


## 4. Model Training — Logistic Regression

Train a Logistic Regression model as a baseline for predicting late deliveries.

Logistic Regression estimates the probability that an order will arrive late based on its features.

The model will be evaluated on the chronological testing dataset to measure its performance on unseen orders.
### Logistic Regression — Baseline Model

Logistic Regression is a supervised classification algorithm that predicts the probability of a binary outcome.

**Our target variable:**
- `0` = Early or on-time delivery
- `1` = Late delivery

#### Step 1: Calculate a Weighted Score

$$
z = b + w_1x_1 + w_2x_2 + \cdots + w_nx_n
$$

Where:
- $z$ = Weighted score
- $b$ = Intercept (bias)
- $w_i$ = Learned weight for feature $i$
- $x_i$ = Value of feature $i$

Each feature contributes to the score according to its learned weight.

For example, features such as estimated delivery days, order value, and same-state shipping may influence the predicted likelihood of a late delivery.

#### Step 2: Convert the Score into a Probability

Logistic Regression applies the **sigmoid function**:

$$
P(y=1 \mid X) = \frac{1}{1+e^{-z}}
$$

Where:
- $P(y=1 \mid X)$ = Predicted probability of a late delivery given the order's features
- $e$ = Euler's number (approximately 2.718)
- $z$ = Weighted score calculated in Step 1

The sigmoid function converts any score into a probability between 0 and 1.

**Example:**

If the weighted score is $z=2$:

$$
P(y=1 \mid X) = \frac{1}{1+e^{-2}} \approx 0.881
$$

The model estimates an **88.1% probability** that the order will arrive late.

#### Step 3: Classify the Order

By default, Logistic Regression uses a probability threshold of 0.5:

$$
\hat{y} =
\begin{cases}
1 & \text{if } P(y=1 \mid X) \geq 0.5 \\
0 & \text{if } P(y=1 \mid X) < 0.5
\end{cases}
$$

- Probability ≥ 50% → Predict late delivery (`1`)
- Probability < 50% → Predict not late (`0`)

The threshold can be adjusted depending on whether identifying more late deliveries or reducing false alarms is more important.

#### Model Training

- `LogisticRegression()` initializes the classification model.
- `fit(X_train_encoded, y_train)` learns feature weights and the intercept from historical orders.
- `predict(X_test_encoded)` generates binary predictions for unseen orders.
- `predict_proba(X_test_encoded)` returns predicted probabilities.
- `max_iter=1000` sets the maximum number of optimization iterations.

#### Model Evaluation

Because late deliveries account for only about 7.6% of the training dataset, accuracy alone may be misleading.

We will evaluate the model using:
- **Precision:** Of orders predicted late, how many were actually late?
- **Recall:** Of all actual late orders, how many did the model identify?
- **F1-score:** Harmonic mean of precision and recall.
- **Confusion Matrix:** Counts correct predictions and classification errors.

### Confusion Matrix — Format

A confusion matrix compares actual outcomes (rows) with model predictions (columns).

| Actual / Predicted | Not Late (0) | Late (1) |
|---|---|---|
| **Not Late (0)** | True Negative (TN) | False Positive (FP) |
| **Late (1)** | False Negative (FN) | True Positive (TP) |

- **TN:** Correctly predicted not late.
- **TP:** Correctly predicted late.
- **FP:** Incorrectly predicted late (false alarm).
- **FN:** Incorrectly predicted not late (missed late delivery).

These metrics help determine whether the model can meaningfully identify orders at risk of late delivery.

In [16]:
from sklearn.linear_model import LogisticRegression

# Create a Logistic Regression model
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model using historical orders and their delivery outcomes
model.fit(X_train_encoded, y_train)

# Predict whether orders in the testing dataset will be late
y_pred = model.predict(X_test_encoded)

# Check how many orders the model predicts will be late
print(pd.Series(y_pred).value_counts())

0    19293
1        3
Name: count, dtype: int64


In [17]:
from sklearn.metrics import classification_report, confusion_matrix

# Compare the model's predictions against actual delivery outcomes
print(classification_report(y_test, y_pred, zero_division=0, digits=4)) # zero_division=0 → Returns 0 when a metric would otherwise be undefined.

# Show the number of correct and incorrect classifications
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0     0.9651    0.9999    0.9822     18622
           1     0.3333    0.0015    0.0030       674

    accuracy                         0.9650     19296
   macro avg     0.6492    0.5007    0.4926     19296
weighted avg     0.9430    0.9650    0.9480     19296

Confusion Matrix:
[[18620     2]
 [  673     1]]


### Baseline Logistic Regression — Results

**Accuracy:** 96.5%

| Metric | Late Deliveries (1) |
|---|---:|
| Precision | 33.3% |
| Recall | 0.15% |
| F1-score | 0.003 |

**Confusion Matrix:**
- True Positives: 1
- True Negatives: 18,620
- False Positives: 2
- False Negatives: 673

#### Why Is the Model Biased?

The training dataset is **imbalanced**, with only 7.6% late deliveries.

As a result, the model assigns low late-delivery probabilities to most orders. Using the default 50% threshold, it predicts almost everything as not late.

Despite 96.5% accuracy, the model identified only **1 of 674 late deliveries**, making it ineffective for our goal.

**Next Step:** Use `class_weight="balanced"`. It tells Logistic Regression to give more importance to the minority class during training. This means the model receives a larger penalty for misclassifying late deliveries, helping it pay more attention to them.

In [18]:
balanced_model = LogisticRegression(
    max_iter=1000, # Allows the model up to 1,000 iterations to optimize its weights and converge during training.
    class_weight="balanced",
    random_state=42
)

# Train the balanced model
balanced_model.fit(X_train_encoded, y_train)

# Predict and evaluate
y_pred_balanced = balanced_model.predict(X_test_encoded)

print(classification_report(
    y_test, y_pred_balanced,
    zero_division=0, digits=4
))

print(confusion_matrix(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0     0.9856    0.5750    0.7263     18622
           1     0.0614    0.7685    0.1138       674

    accuracy                         0.5818     19296
   macro avg     0.5235    0.6718    0.4200     19296
weighted avg     0.9534    0.5818    0.7049     19296

[[10708  7914]
 [  156   518]]


In [19]:
# Get predicted probabilities for late deliveries (class 1)
y_prob_balanced = balanced_model.predict_proba(X_test_encoded)[:, 1]

# Try a higher probability threshold
threshold = 0.7

# Predict late only when probability reaches the threshold
y_pred_threshold = (y_prob_balanced >= threshold).astype(int)

print(classification_report(
    y_test, y_pred_threshold,
    zero_division=0,
    digits=4
))

print(confusion_matrix(y_test, y_pred_threshold))

              precision    recall  f1-score   support

           0     0.9699    0.8938    0.9303     18622
           1     0.0740    0.2344    0.1125       674

    accuracy                         0.8708     19296
   macro avg     0.5220    0.5641    0.5214     19296
weighted avg     0.9386    0.8708    0.9018     19296

[[16645  1977]
 [  516   158]]



### Balanced Logistic Regression — Results

Used `class_weight="balanced"` to give late deliveries more importance during training.

| Metric | Threshold 0.5 | Threshold 0.7 |
|---|---:|---:|
| Precision (Late) | 6.14% | 7.40% |
| Recall (Late) | 76.85% | 23.44% |
| F1-score (Late) | 0.1138 | 0.1125 |
| Accuracy | 58.18% | 87.08% |
| False Positives | 7,914 | 1,977 |
| True Positives | 518 | 158 |

**Findings:**
- `class_weight="balanced"` significantly improved recall compared with the baseline model (0.15%).
- Increasing the threshold to `0.7` reduced false alarms but missed more late deliveries.
- Neither threshold provides a strong balance between precision and recall.

### Next Steps

1. **Chronological Validation Split:** Divide the data into training (60%), validation (20%), and testing (20%) sets.
2. **Threshold Optimization:** Use validation data to select a threshold without influencing final test results.
3. **Final Evaluation:** Compare models using precision, recall, F1-score, and the confusion matrix on the untouched test set.

**Goal:** Identify more late deliveries while minimizing unnecessary alerts.

In [ ]:
# Calculate chronological split boundaries
train_end = int(len(df_sorted) * 0.60)
val_end = int(len(df_sorted) * 0.80)

# Split input features
X_train = df_sorted[feature_columns].iloc[:train_end].copy()
X_val = df_sorted[feature_columns].iloc[train_end:val_end].copy()
X_test = df_sorted[feature_columns].iloc[val_end:].copy()

# Split target values using the same boundaries
y_train = df_sorted["is_late"].iloc[:train_end].copy()
y_val = df_sorted["is_late"].iloc[train_end:val_end].copy()
y_test = df_sorted["is_late"].iloc[val_end:].copy()

# Check dataset sizes and late delivery rates
for name, X_split, y_split in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Testing", X_test, y_test)
]:
    print(
        f"{name}: {X_split.shape}, "
        f"Late rate: {y_split.mean():.2%}"
    )

Training: (57885, 10), Late rate: 6.81%
Validation: (19295, 10), Late rate: 9.94%
Testing: (19296, 10), Late rate: 3.49%


In [22]:
# One-hot encode customer states separately for each dataset
X_train_encoded = pd.get_dummies(
    X_train, columns=["customer_state"], dtype=int
)

X_val_encoded = pd.get_dummies(
    X_val, columns=["customer_state"], dtype=int
)

X_test_encoded = pd.get_dummies(
    X_test, columns=["customer_state"], dtype=int
)

# Match validation and testing columns to training columns
X_val_encoded = X_val_encoded.reindex(
    columns=X_train_encoded.columns, fill_value=0
)

X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns, fill_value=0
)

# Verify all three datasets have matching features
print("Training:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Testing:", X_test_encoded.shape)

Training: (57885, 36)
Validation: (19295, 36)
Testing: (19296, 36)


In [23]:
from sklearn.preprocessing import StandardScaler

# Create a new scaler
scaler = StandardScaler()

# Learn the mean and standard deviation from training data
X_train_encoded[numeric_features] = scaler.fit_transform(
    X_train_encoded[numeric_features]
)

# Apply training statistics to validation data
X_val_encoded[numeric_features] = scaler.transform(
    X_val_encoded[numeric_features]
)

# Apply training statistics to testing data
X_test_encoded[numeric_features] = scaler.transform(
    X_test_encoded[numeric_features]
)

# Verify the scaled training features
X_train_encoded[numeric_features].head()

,purchase_month,purchase_day_of_week,purchase_hour,estimated_delivery_days,total_order_value,total_items,total_sellers,total_seller_states
29811,0.666752,0.112864,-0.516590,-0.812531,-0.001475,3.446189,-0.102694,-0.066436
90509,0.924337,-1.419173,-1.073505,-0.173122,-0.511445,-0.262356,-0.102694,-0.066436
27599,0.924337,-1.419173,0.225963,1.166959,-0.550274,-0.262356,-0.102694,-0.066436
89876,0.924337,-1.419173,1.154155,3.400006,-0.552216,-0.262356,-0.102694,-0.066436
95058,0.924337,-1.419173,1.154155,3.899934,-0.479460,-0.262356,-0.102694,-0.066436


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Create a balanced Logistic Regression model
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train using only the training dataset
balanced_model.fit(X_train_encoded, y_train)

# Predict outcomes for the validation dataset
y_val_pred = balanced_model.predict(X_val_encoded)

# Evaluate model performance
print(classification_report(
    y_val, y_val_pred,
    zero_division=0,
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

              precision    recall  f1-score   support

           0     0.9441    0.6527    0.7718     17378
           1     0.1710    0.6495    0.2707      1917

    accuracy                         0.6523     19295
   macro avg     0.5575    0.6511    0.5212     19295
weighted avg     0.8673    0.6523    0.7220     19295

Confusion Matrix:
[[11342  6036]
 [  672  1245]]


In [37]:
# Sort orders chronologically
df_sorted = df.sort_values("order_purchase_timestamp")

# Calculate split boundaries
train_end = int(len(df_sorted) * 0.60)
val_end = int(len(df_sorted) * 0.80)

# Split features
X_train = df_sorted[feature_columns].iloc[:train_end].copy()
X_val = df_sorted[feature_columns].iloc[train_end:val_end].copy()
X_test = df_sorted[feature_columns].iloc[val_end:].copy()

# Split target variable
y_train = df_sorted["is_late"].iloc[:train_end].copy()
y_val = df_sorted["is_late"].iloc[train_end:val_end].copy()
y_test = df_sorted["is_late"].iloc[val_end:].copy()

# Verify dataset sizes and late-delivery rates
for name, X, y in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Testing", X_test, y_test)
]:
    print(
        f"{name}: {X.shape} | "
        f"Late delivery rate: {y.mean():.2%}"
    )

Training: (57885, 10) | Late delivery rate: 6.81%
Validation: (19295, 10) | Late delivery rate: 9.94%
Testing: (19296, 10) | Late delivery rate: 3.49%


In [38]:
# One-hot encode customer_state
X_train_encoded = pd.get_dummies(
    X_train, columns=["customer_state"], dtype=int
)

X_val_encoded = pd.get_dummies(
    X_val, columns=["customer_state"], dtype=int
)

X_test_encoded = pd.get_dummies(
    X_test, columns=["customer_state"], dtype=int
)

# Ensure validation and testing have the same columns as training
X_val_encoded = X_val_encoded.reindex(
    columns=X_train_encoded.columns, fill_value=0
)

X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns, fill_value=0
)

# Verify shapes
print("Training:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Testing:", X_test_encoded.shape)

Training: (57885, 36)
Validation: (19295, 36)
Testing: (19296, 36)


In [39]:
from sklearn.preprocessing import StandardScaler

# Numerical features to scale
numeric_features = [
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "estimated_delivery_days",
    "total_order_value",
    "total_items",
    "total_sellers",
    "total_seller_states"
]

# Initialize scaler
scaler = StandardScaler()

# Fit scaler ONLY on training data
X_train_encoded[numeric_features] = scaler.fit_transform(
    X_train_encoded[numeric_features]
)

# Apply training scaling to validation and testing
X_val_encoded[numeric_features] = scaler.transform(
    X_val_encoded[numeric_features]
)

X_test_encoded[numeric_features] = scaler.transform(
    X_test_encoded[numeric_features]
)

# Verify scaled values
print(X_train_encoded[numeric_features].head())

       purchase_month  purchase_day_of_week  purchase_hour  \
29811        0.666752              0.112864      -0.516590   
90509        0.924337             -1.419173      -1.073505   
27599        0.924337             -1.419173       0.225963   
89876        0.924337             -1.419173       1.154155   
95058        0.924337             -1.419173       1.154155   

       estimated_delivery_days  total_order_value  total_items  total_sellers  \
29811                -0.812531          -0.001475     3.446189      -0.102694   
90509                -0.173122          -0.511445    -0.262356      -0.102694   
27599                 1.166959          -0.550274    -0.262356      -0.102694   
89876                 3.400006          -0.552216    -0.262356      -0.102694   
95058                 3.899934          -0.479460    -0.262356      -0.102694   

       total_seller_states  
29811            -0.066436  
90509            -0.066436  
27599            -0.066436  
89876            -0.0664

In [40]:
from sklearn.linear_model import LogisticRegression

# Model 1: Standard Logistic Regression
standard_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Model 2: Balanced Logistic Regression
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train both models on the same dataset
standard_model.fit(X_train_encoded, y_train)
balanced_model.fit(X_train_encoded, y_train)

print("Both models trained successfully!")

Both models trained successfully!


In [41]:
from sklearn.metrics import classification_report, confusion_matrix

# Generate validation predictions
standard_val_pred = standard_model.predict(X_val_encoded)
balanced_val_pred = balanced_model.predict(X_val_encoded)

# Evaluate Standard Logistic Regression
print("STANDARD LOGISTIC REGRESSION")
print(classification_report(
    y_val, standard_val_pred,
    digits=4, zero_division=0
))
print(confusion_matrix(y_val, standard_val_pred))

# Evaluate Balanced Logistic Regression
print("\nBALANCED LOGISTIC REGRESSION")
print(classification_report(
    y_val, balanced_val_pred,
    digits=4, zero_division=0
))
print(confusion_matrix(y_val, balanced_val_pred))

STANDARD LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0     0.9006    1.0000    0.9477     17378
           1     0.0000    0.0000    0.0000      1917

    accuracy                         0.9006     19295
   macro avg     0.4503    0.5000    0.4739     19295
weighted avg     0.8112    0.9006    0.8536     19295

[[17378     0]
 [ 1917     0]]

BALANCED LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0     0.9441    0.6527    0.7718     17378
           1     0.1710    0.6495    0.2707      1917

    accuracy                         0.6523     19295
   macro avg     0.5575    0.6511    0.5212     19295
weighted avg     0.8673    0.6523    0.7220     19295

[[11342  6036]
 [  672  1245]]



## Logistic Regression — Model Evaluation

### Dataset Split
Used a **60/20/20 chronological split**:
- **60% Training:** Train models.
- **20% Validation:** Evaluate and improve predictions.
- **20% Testing:** Assess performance on future orders.

### Model Results (Validation)

| Metric | Standard Logistic Regression | Balanced Logistic Regression |
|---|---:|---:|
| Accuracy | 90.06% | 65.23% |
| Late Precision | 0.00% | 17.10% |
| Late Recall | 0.00% | 64.95% |
| Late F1-score | 0.0000 | 0.2707 |

**Findings:**
- Standard Logistic Regression predicted every order as on-time due to class imbalance.
- Balanced Logistic Regression correctly identified **1,245 of 1,917 late deliveries**, but generated 6,036 false positives.
- Balanced Logistic Regression is more useful for identifying delivery risks despite its lower accuracy.

### New Approach: Three-Level Delivery Risk System

Instead of predicting only **Late** or **On Time**, we will use Balanced Logistic Regression to assign each order a risk category.

| Risk Level | Model Score | Business Action |
|---|---|---|
| Low Risk | Below 0.30 | Normal processing |
| Medium Risk | 0.30–0.69 | Monitor delivery |
| High Risk | 0.70 or higher | Prioritize investigation |

### Objective
Evaluate the **actual late-delivery rate** within each risk category to determine whether the model can meaningfully distinguish low-risk from high-risk orders.

**Note:** Thresholds are experimental, and model scores are not necessarily calibrated probabilities of late delivery.


In [42]:
import numpy as np

# Get predicted risk scores from the balanced model
risk_scores = balanced_model.predict_proba(X_val_encoded)[:, 1]

# Assign risk levels based on the scores
risk_levels = np.select(
    [
        risk_scores < 0.30,
        risk_scores < 0.70
    ],
    [
        "Low Risk",
        "Medium Risk"
    ],
    default="High Risk"
)

# Create a DataFrame to evaluate the risk categories
risk_analysis = pd.DataFrame({
    "risk_score": risk_scores,
    "risk_level": risk_levels,
    "is_late": y_val.to_numpy()
})

# Calculate actual late-delivery rates for each category
risk_summary = (
    risk_analysis.groupby("risk_level")
    .agg(
        total_orders=("is_late", "count"),
        late_orders=("is_late", "sum"),
        actual_late_rate=("is_late", "mean")
    )
    .reindex(["Low Risk", "Medium Risk", "High Risk"])
)

risk_summary["actual_late_rate"] *= 100

print(risk_summary.round(2))

             total_orders  late_orders  actual_late_rate
risk_level                                              
Low Risk             1267           15              1.18
Medium Risk         16489         1443              8.75
High Risk            1539          459             29.82


In [43]:
# Generate risk scores for testing data
test_risk_scores = balanced_model.predict_proba(X_test_encoded)[:, 1]

# Assign risk categories using the same thresholds
test_risk_levels = np.select(
    [
        test_risk_scores < 0.30,
        test_risk_scores < 0.70
    ],
    [
        "Low Risk",
        "Medium Risk"
    ],
    default="High Risk"
)

# Create testing analysis DataFrame
test_risk_analysis = pd.DataFrame({
    "risk_score": test_risk_scores,
    "risk_level": test_risk_levels,
    "is_late": y_test.to_numpy()
})

# Calculate actual late-delivery rates
test_risk_summary = (
    test_risk_analysis.groupby("risk_level")
    .agg(
        total_orders=("is_late", "count"),
        late_orders=("is_late", "sum"),
        actual_late_rate=("is_late", "mean")
    )
    .reindex(["Low Risk", "Medium Risk", "High Risk"])
)

test_risk_summary["actual_late_rate"] *= 100

print(test_risk_summary.round(2))

             total_orders  late_orders  actual_late_rate
risk_level                                              
Low Risk             2398            8              0.33
Medium Risk         15063          532              3.53
High Risk            1835          134              7.30


## Final Model: Three-Level Delivery Risk System

Used **Balanced Logistic Regression** to classify orders into three risk categories based on predicted scores.

### Risk Categories
- **Low Risk:** Score below 0.30
- **Medium Risk:** Score between 0.30 and 0.70
- **High Risk:** Score of 0.70 or higher

### Final Results

| Risk Level | Validation Late Rate | Testing Late Rate |
|---|---:|---:|
| Low Risk | 1.18% | 0.33% |
| Medium Risk | 8.75% | 3.53% |
| High Risk | 29.82% | 7.30% |

### Key Findings
- Late-delivery rates increased consistently from Low to High Risk.
- High-risk orders were approximately **22 times more likely to arrive late** than low-risk orders in testing.
- The model demonstrated useful risk separation on newer orders, although absolute late-delivery rates changed.
- Most orders fell into Medium Risk, limiting how precisely the system prioritizes orders.

### Conclusion
The model is more useful for **delivery risk prioritization** than predicting individual late deliveries.

Businesses could use these categories to monitor shipments and prioritize high-risk orders for investigation.

**Limitation:** Risk scores are not calibrated probabilities, and the model requires further validation before real-world deployment.


In [21]:
# Close all idle database connections managed by the engine
engine.dispose()